In [ ]:
# hi

In [25]:
from tensorflow.keras.datasets import boston_housing
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers


In [19]:
(train_data, train_targets), (test_data, test_targets) = (boston_housing.load_data())

In [22]:
# normalization:
mean = train_data.mean(axis=0)
train_data -= mean
std = train_data.std(axis=0)
train_data /= std
test_data -= mean
test_data /= std

In [23]:
def build_model():
  model = keras.Sequential([layers.Dense(64, activation="relu"),
                            layers.Dense(64, activation="relu"),
                            layers.Dense(1)])
  model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
  return model

## Mean Absolute Error (MAE)

For one observation:

$$
AE(y, \hat{y}) = |y - \hat{y}|
$$

For \(N\) observations:

$$
MAE = \frac{1}{N} \sum_{i=1}^{N} |y_i - \hat{y}_i|
$$

### Parameters

- \(MAE\): mean absolute error
- \(AE\): absolute error for one observation
- \(N\): number of observations
- \(y\): true value for one observation
- \(\hat{y}\): predicted value for one observation
- \(y_i\): true value of observation \(i\)
- \(\hat{y}_i\): predicted value of observation \(i\)
- \(|\cdot|\): absolute value
- \(i\): observation index, from \(1\) to \(N\)

## Mean Squared Error (MSE)

For one observation:

$$
SE(y, \hat{y}) = (y - \hat{y})^2
$$

For \(N\) observations:

$$
MSE = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2
$$

### Parameters

- \(MSE\): mean squared error
- \(SE\): squared error for one observation
- \(N\): number of observations
- \(y\): true value for one observation
- \(\hat{y}\): predicted value for one observation
- \(y_i\): true value of observation \(i\)
- \(\hat{y}_i\): predicted value of observation \(i\)
- \(i\): observation index, from \(1\) to \(N\)

In [29]:
k = 4
num_val_samples = len(train_data) // k
num_epochs = 100
all_scores = []


for i in range(k):
  print(f"Processing fold #{i}")
  val_data = train_data[i * num_val_samples: (i + 1) * num_val_samples]
  val_targets = train_targets[i * num_val_samples: (i + 1) * num_val_samples]
  partial_train_data = np.concatenate([train_data[:i * num_val_samples],
                                      train_data[(i + 1) * num_val_samples:]],
                                      axis=0)
  partial_train_targets = np.concatenate([train_targets[:i * num_val_samples],
                                          train_targets[(i + 1) * num_val_samples:]],
                                          axis=0)
  model = build_model()

  model.fit(partial_train_data,
            partial_train_targets,
            epochs=num_epochs, batch_size=16, verbose=1)

  val_mse, val_mae = model.evaluate(val_data, val_targets, verbose=1)

  all_scores.append(val_mae)

Processing fold #0
Epoch 1/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 511.0063 - mae: 20.6491   
Epoch 2/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 378.5119 - mae: 17.1791 
Epoch 3/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 236.0465 - mae: 12.8737 
Epoch 4/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 128.0805 - mae: 8.9220 
Epoch 5/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 74.0592 - mae: 6.6360  
Epoch 6/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 48.0340 - mae: 5.2798 
Epoch 7/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 34.1367 - mae: 4.3793 
Epoch 8/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 26.4479 - mae: 3.7583 
Epoch 9/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 22.6174 - mae: 3.4239 
Epoch 10/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 20.2394 - mae: 3.1697 
Epoch 11/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 18.6522 - mae: 3.0533
Epoch 12/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 16.9510 

In [30]:
all_scores
# mae in all val folds

[2.0553998947143555, 2.448786973953247, 2.4781813621520996, 2.356452226638794]

In [31]:
np.mean(all_scores)

np.float64(2.334705114364624)

In [32]:
num_epochs = 500
all_mae_histories = []

for i in range(k):
  print(f"Processing fold #{i}")
  val_data = train_data[i * num_val_samples: (i + 1) * num_val_samples]
  val_targets = train_targets[i * num_val_samples: (i + 1) * num_val_samples]
  partial_train_data = np.concatenate([train_data[:i * num_val_samples],
                                       train_data[(i + 1) * num_val_samples:]],
                                       axis=0)
  partial_train_targets = np.concatenate([train_targets[:i * num_val_samples],
                                          train_targets[(i + 1) * num_val_samples:]],
                                          axis=0)
  model = build_model()

  history = model.fit(partial_train_data,
                      partial_train_targets,
                      validation_data=(val_data, val_targets),
                      epochs=num_epochs, batch_size=16, verbose=0)

  mae_history = history.history["val_mae"]
  all_mae_histories.append(mae_history)

Processing fold #0


KeyboardInterrupt: 